# FZ Advanced Features: Caching, I/O, and Administration

Topics covered:

1. **Cache calculator** — reuse previous results without re-running
2. **Numpy array inputs** — pass arrays from numpy/scipy DOE generators
3. **Multi-valued outputs** — multiple output variables in one run
4. **Logging & configuration** — verbosity, interpreter, `print_config`
5. **`fzl` with check** — validate models and calculators
6. **Install / uninstall** — manage model plugins
7. **`fzd` with `algorithm_options`** — fine-tune algorithm behaviour
8. **Coarse-to-fine workflow** — combine cache + adaptive DOE


In [1]:
import json, time, shutil
from pathlib import Path
import fz
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Start fresh so fzd cache never picks up stale results
WORK = Path("work_05_advanced")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir()
(WORK / "models").mkdir()

fz.set_log_level("WARNING")
print("fz", fz.__version__)


fz 1.0


## 1 · Cache calculator

The `cache://` calculator matches cases by MD5 hash of their compiled input files. If a match exists, it returns the stored result instantly.

In [2]:
SIM_SLOW = WORK / "models" / "slow_sin.py"
SIM_SLOW.write_text('''#!/usr/bin/env python3
import time, math
params = {}
with open("in.txt") as fh:
    for line in fh:
        line = line.split("#")[0].strip()
        if "=" in line:
            k, v = line.split("=", 1)
            params[k.strip()] = float(v.strip())
x = params["x"]
time.sleep(0.3)    # simulate computation time
y = math.sin(x) + 0.5 * math.sin(3*x)
with open("output.txt", "w") as fh:
    fh.write(f"y = {y:.10f}\\n")
''')

TMPL_S = WORK / "in.txt"
TMPL_S.write_text("x = ${x~0.0}\n")
MODEL_S = {
    "varprefix": "$", "delim": "{}", "commentline": "#",
    "output": {"y": "grep '^y = ' output.txt | awk '{print $3}'"},
}
CACHE_DIR = str(WORK / "results_cache")
CALC_REAL  = f"sh://python3 {SIM_SLOW.resolve()}"
xs = list(np.linspace(0, 2*np.pi, 8))

# --- Cold run ---
t0 = time.time()
df1 = fz.fzr(str(TMPL_S), {"x": xs}, MODEL_S,
              results_dir=CACHE_DIR, calculators=[CALC_REAL])
cold = time.time() - t0
print(f"Cold run:  {cold:.2f}s  ({len(df1)} cases)")

# --- Warm run via cache:// ---
t0 = time.time()
df2 = fz.fzr(str(TMPL_S), {"x": xs}, MODEL_S,
              results_dir=str(WORK / "results_from_cache"),
              calculators=[f"cache://{CACHE_DIR}", CALC_REAL])
warm = time.time() - t0
print(f"Warm run:  {warm:.2f}s   speedup: {cold/max(warm,0.001):.1f}x")

df1["y"] = pd.to_numeric(df1["y"], errors="coerce")
df2["y"] = pd.to_numeric(df2["y"], errors="coerce")
diff = abs(df1["y"].sort_values().values - df2["y"].sort_values().values).max()
print(f"Max value diff (cold vs cached): {diff:.2e}  ✓")


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/8) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/8) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/8) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/8) ETA: ...

◢ [█████>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  12% (1/8) ETA: 3s

...

  [████████████████████████████████████████] 100% (8/8) Total: 4s

Cold run:  4.33s  (8 cases)
  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/8) ETA: ...

Cache match found in subdirectory: work_05_advanced/results_cache/x=0.0
Cache match found in subdirectory: work_05_advanced/results_cache/x=0.8975979010256552
Cache match found in subdirectory: work_05_advanced/results_cache/x=1.7951958020513104
Cache match found in subdirectory: work_05_advanced/results_cache/x=2.6927937030769655
Cache match found in subdirectory: work_05_advanced/results_cache/x=3.5903916041026207
Cache match found in subdirectory: work_05_advanced/results_cache/x=4.487989505128276
Cache match found in subdirectory: work_05_advanced/results_cache/x=5.385587406153931
Cache match found in subdirectory: work_05_advanced/results_cache/x=6.283185307179586


  [████████████████████████████████████████] 100% (8/8) Total: 0s

Warm run:  0.26s   speedup: 16.6x
Max value diff (cold vs cached): 0.00e+00  ✓


## 2 · Numpy array inputs

fzr accepts `numpy.ndarray` values directly inside `input_variables`.

In [3]:
rng = np.random.default_rng(42)
xs_np = rng.uniform(0, 2*np.pi, size=20)   # numpy array

df_np = fz.fzr(
    str(TMPL_S),
    {"x": xs_np},
    MODEL_S,
    results_dir=str(WORK / "results_numpy"),
    calculators=[CALC_REAL],
)
df_np["x"] = df_np["x"].astype(float)
df_np["y"] = pd.to_numeric(df_np["y"], errors="coerce")
print(f"numpy input: {len(xs_np)} points  |  output shape: {df_np.shape}")
print(df_np[["x", "y"]].sort_values("x").to_string(index=False))


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/20) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/20) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/20) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/20) ETA: ...

◢ [██>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   5% (1/20) ETA: 9s

...

◥ [██████████████████████████████████████>░]  95% (19/20) ETA: 0s

◢ [██████████████████████████████████████>░]  95% (19/20) ETA: 0s

◣ [██████████████████████████████████████>░]  95% (19/20) ETA: 0s

  [████████████████████████████████████████] 100% (20/20) Total: 10s

numpy input: 20 points  |  output shape: (20, 7)
       x         y
0.400976  0.856865
0.591734  1.047392
0.804962  1.053010
1.427783  0.535108
2.329793  1.049998
2.757555  0.831480
2.786054  0.785881
2.829858  0.709070
3.484559 -0.764648
3.968864 -1.042558
4.045524 -0.994109
4.381693 -0.672345
4.782382 -0.508534
4.862909 -0.538810
4.938988 -0.585586
5.169564 -0.798325
5.200160 -0.829741
5.394730 -1.005313
5.823036 -0.935051
6.130016 -0.374325


## 3 · Multiple output variables

A model can extract several outputs from one run.  All become columns in the result DataFrame.

In [4]:
SIM_MULTI = WORK / "models" / "multi_out.py"
SIM_MULTI.write_text('''#!/usr/bin/env python3
import math
params = {}
with open("multi.in") as fh:
    for line in fh:
        line = line.split("#")[0].strip()
        if "=" in line:
            k, v = line.split("=", 1)
            params[k.strip()] = float(v.strip())
x = params["x"]
with open("output.txt", "w") as fh:
    fh.write(f"sin_x     = {math.sin(x):.8f}\\n")
    fh.write(f"cos_x     = {math.cos(x):.8f}\\n")
    fh.write(f"x_squared = {x**2:.8f}\\n")
    fh.write(f"abs_val   = {abs(x):.8f}\\n")
''')

TMPL_M = WORK / "multi.in"
TMPL_M.write_text("x = ${x~0.0}\n")
MODEL_MULTI = {
    "varprefix": "$", "delim": "{}", "commentline": "#",
    "output": {
        "sin_x":     "grep '^sin_x'     output.txt | awk '{print $3}'",
        "cos_x":     "grep '^cos_x'     output.txt | awk '{print $3}'",
        "x_squared": "grep '^x_squared' output.txt | awk '{print $3}'",
        "abs_val":   "grep '^abs_val'   output.txt | awk '{print $3}'",
    },
}
CALC_M = f"sh://python3 {SIM_MULTI.resolve()}"

df_m = fz.fzr(
    str(TMPL_M),
    {"x": list(np.linspace(-np.pi, np.pi, 9))},
    MODEL_MULTI,
    results_dir=str(WORK / "results_multi"),
    calculators=[CALC_M],
)
for col in ["sin_x", "cos_x", "x_squared", "abs_val"]:
    df_m[col] = pd.to_numeric(df_m[col], errors="coerce")
df_m["x"] = df_m["x"].astype(float)
print("Output columns:", [c for c in df_m.columns if c not in ("path","calculator","status","error","command")])
print(df_m[["x", "sin_x", "cos_x", "x_squared"]].sort_values("x").to_string(index=False))


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/9) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/9) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/9) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/9) ETA: ...

◢ [████>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  11% (1/9) ETA: 4s

...

◣ [███████████████████████████████████>░░░░]  88% (8/9) ETA: 0s

◤ [███████████████████████████████████>░░░░]  88% (8/9) ETA: 0s

◥ [███████████████████████████████████>░░░░]  88% (8/9) ETA: 0s

  [████████████████████████████████████████] 100% (9/9) Total: 4s

Output columns: ['x', 'sin_x', 'cos_x', 'x_squared', 'abs_val']
        x     sin_x     cos_x  x_squared
-3.141593 -0.000000 -1.000000   9.869604
-2.356194 -0.707107 -0.707107   5.551652
-1.570796 -1.000000  0.000000   2.467401
-0.785398 -0.707107  0.707107   0.616850
 0.000000  0.000000  1.000000   0.000000
 0.785398  0.707107  0.707107   0.616850
 1.570796  1.000000  0.000000   2.467401
 2.356194  0.707107 -0.707107   5.551652
 3.141593  0.000000 -1.000000   9.869604


## 4 · Logging & configuration

In [5]:
print("=== Default config ===")
fz.print_config()

print("\n=== Log level cycle ===")
for level in ("QUIET", "ERROR", "WARNING", "INFO", "DEBUG"):
    fz.set_log_level(level)
    print(f"  set({level!r}) → get() = {fz.get_log_level()}")
fz.set_log_level("WARNING")  # reset

print("\n=== Interpreter ===")
fz.set_interpreter("python")
print(f"  interpreter: {fz.get_interpreter()}")

print("\n=== Config object attrs ===")
cfg = fz.get_config()
print([a for a in dir(cfg) if not a.startswith("_")])


=== Default config ===
FZ PACKAGE CONFIGURATION
Version: 1.0
Commit:  (2026-04-27)

🔧 LOGGING:
  FZ_LOG_LEVEL = WARNING

🔄 CALCULATION:
  FZ_MAX_RETRIES = 5
  FZ_INTERPRETER = python
  Current interpreter = python

⚡ PERFORMANCE:
  FZ_MAX_WORKERS = auto

🌐 SSH:
  FZ_SSH_AUTO_ACCEPT_HOSTKEYS = False
  FZ_SSH_KEEPALIVE = 300s

⏱️  RUN TIMEOUT:
  FZ_RUN_TIMEOUT = 600s

🔍 SHELL PATH:
  FZ_SHELL_PATH = (not set, use system PATH)

Set environment variables to customize these defaults
Example: export FZ_LOG_LEVEL=INFO FZ_MAX_RETRIES=3

=== Log level cycle ===
  set('QUIET') → get() = LogLevel.QUIET
  set('ERROR') → get() = LogLevel.ERROR
  set('WARNING') → get() = LogLevel.WARNING
  set('INFO') → get() = LogLevel.INFO
  set('DEBUG') → get() = LogLevel.DEBUG

=== Interpreter ===
  interpreter: python

=== Config object attrs ===
['get_summary', 'interpreter', 'log_level', 'max_retries', 'max_workers', 'reload', 'run_timeout', 'shell_path', 'ssh_auto_accept_hostkeys', 'ssh_keepalive']


## 5 · `fzl` with `check=True`

The `check` flag validates each model and calculator.

In [6]:
info = fz.fzl(check=True)
print("Models:")
for name, props in info["models"].items():
    status = props.get("check_status", "?")
    print(f"  {name:25s}  {status}")
print("\nCalculators:")
for name, props in info["calculators"].items():
    status = props.get("check_status", "?")
    print(f"  {name:35s}  {status}")


Models:
  Moret                      passed

Calculators:
  sh://                                failed


## 6 · Install and uninstall a model plugin

In [7]:
print("Installing fz-moret model (from GitHub)...")
try:
    result = fz.install_model("moret", global_install=False)
    print(f"  Installed: model_name={result['model_name']}")
    print(f"             install_path={result['install_path']}")
except Exception as e:
    print(f"  Note: {e}")

print("\nModels after install:")
for name in fz.fzl()["models"]:
    print(f"  {name}")

print("\nUninstalling Moret (local)...")
try:
    fz.uninstall_model("Moret", global_uninstall=False)
    print("  Done.")
except Exception as e:
    print(f"  {e}")

print("\nModels after uninstall:")
for name in fz.fzl()["models"]:
    print(f"  {name}")


Installing fz-moret model (from GitHub)...


  Installed: model_name=Moret
             install_path=/home/richet/Sync/Open/Funz/github/fz/examples/.fz/models/Moret.json

Models after install:
  Moret

Uninstalling Moret (local)...
  Done.

Models after uninstall:
  Moret


## 7 · `fzd` algorithm_options

Same algorithm, different `nvalues` → different sample sizes.

In [8]:
ALGO_RS_PATH = None
for p in [Path("../algorithms/randomsampling.py"),
          Path("/home/richet/Sync/Open/Funz/github/fz/examples/algorithms/randomsampling.py")]:
    if p.exists():
        ALGO_RS_PATH = str(p)
        break

results_table = []
for n in [10, 25, 50]:
    res = fz.fzd(
        str(TMPL_S),
        {"x": "[0;6.283]"},
        MODEL_S,
        output_expression="y",
        algorithm=ALGO_RS_PATH,
        algorithm_options={"nvalues": n, "seed": 99},
        calculators=[CALC_REAL],
        analysis_dir=str(WORK / f"doe_rs_{n}"),
    )
    df = res["XY"]
    y_vals = df["y"].astype(float)
    results_table.append({"nvalues": n, "n_pts": len(df),
                           "y_mean": y_vals.mean(), "y_std": y_vals.std()})
    print(f"  nvalues={n:3d}  → {len(df)} points, "
          f"y_mean={y_vals.mean():.4f}, y_std={y_vals.std():.4f}")


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/10) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/10) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/10) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/10) ETA: ...

◢ [████>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  10% (1/10) ETA: 4s

...

◤ [███████████████████████████████████████>]  98% (49/50) ETA: 0s

◥ [███████████████████████████████████████>]  98% (49/50) ETA: 0s

◢ [███████████████████████████████████████>]  98% (49/50) ETA: 0s

  [████████████████████████████████████████] 100% (50/50) Total: 28s

  nvalues= 50  → 50 points, y_mean=0.0143, y_std=0.7813


## 8 · Coarse-to-fine exploration + cache reuse

In [9]:
# Step 1: coarse exploration
res_coarse = fz.fzd(
    str(TMPL_S), {"x": "[0;6.283]"}, MODEL_S,
    output_expression="y",
    algorithm=ALGO_RS_PATH,
    algorithm_options={"nvalues": 15, "seed": 1},
    calculators=[CALC_REAL],
    analysis_dir=str(WORK / "explore_coarse"),
)
df_coarse = res_coarse["XY"].copy()
df_coarse["x"] = df_coarse["x"].astype(float)
df_coarse["y"] = df_coarse["y"].astype(float)

# Step 2: refine around peak
x_peak = float(df_coarse.loc[df_coarse["y"].idxmax(), "x"])
x_lo   = max(0.0, x_peak - 0.5)
x_hi   = min(2*np.pi, x_peak + 0.5)
print(f"Coarse peak at x ≈ {x_peak:.3f} → refine in [{x_lo:.2f}, {x_hi:.2f}]")

res_fine = fz.fzd(
    str(TMPL_S), {"x": f"[{x_lo};{x_hi}]"}, MODEL_S,
    output_expression="y",
    algorithm=ALGO_RS_PATH,
    algorithm_options={"nvalues": 20, "seed": 2},
    # cache:// avoids re-running any already-computed cases
    calculators=[f"cache://{WORK / 'explore_coarse'}", CALC_REAL],
    analysis_dir=str(WORK / "explore_fine"),
)
df_fine = res_fine["XY"].copy()
df_fine["x"] = df_fine["x"].astype(float)
df_fine["y"] = df_fine["y"].astype(float)

xs_true = np.linspace(0, 2*np.pi, 500)
ys_true = np.sin(xs_true) + 0.5*np.sin(3*xs_true)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(xs_true, ys_true, "k-", lw=2, label="true curve")
ax.scatter(df_coarse["x"], df_coarse["y"], c="steelblue", s=40,
           label="coarse samples", zorder=5)
ax.scatter(df_fine["x"],   df_fine["y"],   c="tomato", s=60, marker="^",
           label="fine samples", zorder=6)
ax.axvline(x_peak, color="green", ls="--", label=f"peak x≈{x_peak:.2f}")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("Coarse-to-fine exploration with cache reuse")
plt.tight_layout()
plt.savefig(str(WORK / "coarse_to_fine.png"), dpi=100)
plt.show()
print("Plot saved.")


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/15) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/15) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/15) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/15) ETA: ...

◢ [██>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   6% (1/15) ETA: 7s

...

◢ [██████████████████████████████████████>░]  95% (19/20) ETA: 0s

◣ [██████████████████████████████████████>░]  95% (19/20) ETA: 0s

  [████████████████████████████████████████] 100% (20/20) Total: 11s

Plot saved.


/tmp/ipykernel_47951/986620178.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
